In [2]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import json
import seaborn as sns

PATH_TO_CONSTANTS = "../../"

with open(PATH_TO_CONSTANTS +"constants.json") as f:
    CONSTANTS = json.load(f)



In [3]:
file_path = PATH_TO_CONSTANTS + CONSTANTS['oregano_v3']

# Create a directed graph
G = nx.DiGraph()

# Read file and add edges
with open(file_path, "r") as f:
    for line in f:
        subj, pred, obj = line.strip().split("\t")  # Split by tab
        G.add_edge(subj, obj, label=pred)  # Store predicate as edge label


In [4]:

# Print graph stats
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 150119
Edges: 3748333


In [8]:
G.edges['NATURAL_COMPOUND:16168', 'PROTEIN:358']

{'label': 'has_target'}

In [ ]:

# Load nodes and edges CSV files
nodes_file = PATH_TO_CONSTANTS + CONSTANTS['cleaned_nodes_csv']
edges_file = PATH_TO_CONSTANTS + CONSTANTS['edges_csv']  # Replace with your edges CSV file path

# Load nodes and edges into DataFrames
nodes_df = pd.read_csv(nodes_file)
edges_df = pd.read_csv(edges_file)

edges_df = edges_df[edges_df['type'] != 'has_code'] #drop the has_code stuff
# Create a directed graph
G = nx.DiGraph()


In [ ]:
len(nodes_df) - nodes_df['name'].notna().sum()

In [ ]:
nodes_df_clean = nodes_df[nodes_df['name'].isna()]
# nodes_df_clean.iloc[-1].properties

In [ ]:
nodes_df_clean.groupby('type').count()

#indication protein and side_effet

In [ ]:
nodes_df_clean = nodes_df[nodes_df['name'].notna()]
nodes_df_clean

In [ ]:
nodes_df_clean['name_length'] = nodes_df_clean['name'].apply(len)

result = nodes_df_clean.groupby('type')['name_length'].mean().reset_index()


# Step 2: Group by 'type' and calculate the average length and count
result = nodes_df_clean.groupby('type').agg(
    avg_name_length=('name_length', 'mean'),
    sample_count=('name', 'count')
).reset_index()

print(result)

In [ ]:
nodes_df_clean['name_length'].sort_values()

In [ ]:
# Filter data for name lengths greater than 150
outliers_df = nodes_df_clean[nodes_df_clean['name_length'] > 150]

# Group by 'type' and count the occurrences
outliers_count = outliers_df.groupby('type').size()

# Display the result
print(outliers_count)

# outliers_df

In [ ]:
# Cap name lengths at 150
nodes_df_clean['name_length'] = nodes_df_clean['name'].apply(lambda x: min(len(x), 150))

# Set Seaborn style
sns.set_theme(style="whitegrid")

# Boxen plot for better distribution visualization
plt.figure(figsize=(12, 4.5))
sns.boxenplot(data=nodes_df_clean, x='type', y='name_length', palette="Set2")
plt.title('Distribution of Name Lengths by Type (Capped at 150)', fontsize=18, fontweight='bold')
plt.xlabel('Type', fontsize=16)
plt.ylabel('Name Length', fontsize=16)
plt.xticks(rotation=45, fontsize=13)
plt.yticks(fontsize=13)
plt.ylim(0, 150)
plt.tight_layout()
plt.savefig(PATH_TO_CONSTANTS+CONSTANTS['figures']+"Distribution.pdf", format='pdf')  # Export with high resolution
plt.show()

